## Introduction (LiteLLM Proxy)

This lesson will cover: 
- What is function calling and its use cases 
- How to create a function call using LiteLLM Proxy 
- How to integrate a function call into an application 

### Setup
Create a `.env` file with:
```
LITELLM_BASE_URL=http://localhost:4000
LITELLM_API_KEY=your-litellm-api-key
LITELLM_MODEL=gpt-4o
```

## Understanding Function Calls 

For this lesson, we want to build a feature for our education startup that allows users to use a chatbot to find technical courses. We will recommend courses that fit their skill level, current role and technology of interest.

In [28]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."

In [29]:
prompt1 = f'''
Extract the following information from the text and return ONLY a valid JSON object.
Do not include markdown, explanations, or code fences.

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Extract the following information from the text and return ONLY a valid JSON object.
Do not include markdown, explanations, or code fences.

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''

In [30]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# LiteLLM Proxy provides an OpenAI-compatible API
client = OpenAI(
    base_url=os.environ["LITELLM_BASE_URL"],
    api_key=os.environ["LITELLM_API_KEY"],
)

deployment = os.environ.get("LITELLM_MODEL", "gpt-4o")

In [31]:
openai_response1 = client.chat.completions.create(
 model=deployment,    
 messages = [{'role': 'user', 'content': prompt1}],
 response_format={"type": "json_object"}
)
openai_response1.choices[0].message.content 

'{\n  "name": "Emily Johnson",\n  "major": "Computer Science",\n  "school": "Duke University",\n  "grades": "3.7 GPA",\n  "club": ["Chess Club", "Debate Team"]\n}'

In [32]:
openai_response2 = client.chat.completions.create(
 model=deployment,    
 messages = [{'role': 'user', 'content': prompt2}],
 response_format={"type": "json_object"},
)
openai_response2.choices[0].message.content

'{\n  "name": "Michael Lee",\n  "major": "Computer Science",\n  "school": "Stanford University",\n  "grades": "3.8 GPA",\n  "club": "Robotics Club"\n}'

In [33]:
print(openai_response1.choices[0].message.content)

{
  "name": "Emily Johnson",
  "major": "Computer Science",
  "school": "Duke University",
  "grades": "3.7 GPA",
  "club": ["Chess Club", "Debate Team"]
}


In [34]:
# Loading the response as a JSON object
import json

json_response1 = json.loads(openai_response1.choices[0].message.content)
json_response1

{'name': 'Emily Johnson',
 'major': 'Computer Science',
 'school': 'Duke University',
 'grades': '3.7 GPA',
 'club': ['Chess Club', 'Debate Team']}

In [35]:
# Loading the response as a JSON object
import json

json_response2 = json.loads(openai_response2.choices[0].message.content )
json_response2

{'name': 'Michael Lee',
 'major': 'Computer Science',
 'school': 'Stanford University',
 'grades': '3.8 GPA',
 'club': 'Robotics Club'}

## 2. Creating Your First Function Call 

The process of creating a function call includes 3 main steps: 
1. Calling the Chat Completions API with a list of your functions and a user message 
2. Read the model's response to perform an action ie execute a function or API Call 
3. Make another call to Chat Completions API with the response from your function to use that information to create a response to the user.

In [36]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

In [37]:
functions = [
   {
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]

# Wrap in the modern "tools" format
tools = [{"type": "function", "function": f} for f in functions]

In [38]:
import requests

def search_courses(role, product=None, level=None):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }

    response = requests.get(url, params=params)
    modules = response.json()["modules"]

    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})

    return str(results)

In [39]:
response = client.chat.completions.create(
    model=deployment,
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

print(response.choices[0].message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_pAaauhOT1pPobHkKFnbr6B67', function=Function(arguments='{"role":"student","product":"Azure","level":"beginner"}', name='search_courses'), type='function')], provider_specific_fields={'refusal': None})


In [40]:
response = client.chat.completions.create(
    model=deployment, 
    messages=messages,
    tools=tools, 
    tool_choice="auto",
) 

print(response.choices[0].message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Ml8ZFRZA8ZfFCUtAoIIGsfkF', function=Function(arguments='{"role":"student","product":"Azure","level":"beginner"}', name='search_courses'), type='function')], provider_specific_fields={'refusal': None})


## 3. Integrating Function Calls into an Application

In [41]:
response_message = response.choices[0].message

if response_message.tool_calls:
    tool_call = response_message.tool_calls[0]

    function_args = json.loads(tool_call.function.arguments)

    function_response = search_courses(**function_args)

    print("Output of function call:")
    print(function_response)

    messages.append(response_message)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": function_response,
    })

Output of function call:
[{'title': 'Describe concepts of cryptography', 'url': 'https://learn.microsoft.com/en-us/training/modules/describe-concepts-of-cryptography/?WT.mc_id=api_CatalogApi'}, {'title': 'Host a web application with Azure App Service', 'url': 'https://learn.microsoft.com/en-us/training/modules/host-a-web-app-with-azure-app-service/?WT.mc_id=api_CatalogApi'}, {'title': 'Describe security and compliance concepts', 'url': 'https://learn.microsoft.com/en-us/training/modules/describe-security-concepts-methodologies/?WT.mc_id=api_CatalogApi'}, {'title': 'Create a voice calling web app with Azure Communication Services', 'url': 'https://learn.microsoft.com/en-us/training/modules/communication-services-voice-calling-web-app/?WT.mc_id=api_CatalogApi'}, {'title': 'The Principles of Sustainable Software Engineering', 'url': 'https://learn.microsoft.com/en-us/training/modules/sustainable-software-engineering-overview/?WT.mc_id=api_CatalogApi'}]


In [42]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)

In [43]:
# Check if the model wants to call a function
if response_message.tool_calls:
    tool_call = response_message.tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.function.name)
    print()

    # Call the function
    function_name = tool_call.function.name

    available_functions = {
        "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name]

    function_args = json.loads(tool_call.function.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))

    # Add the assistant response and tool response to the messages
    messages.append(response_message)
    messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": function_response,
        }
    )

Recommended Function call:
search_courses

Output of function call:
[{'title': 'Describe concepts of cryptography', 'url': 'https://learn.microsoft.com/en-us/training/modules/describe-concepts-of-cryptography/?WT.mc_id=api_CatalogApi'}, {'title': 'Host a web application with Azure App Service', 'url': 'https://learn.microsoft.com/en-us/training/modules/host-a-web-app-with-azure-app-service/?WT.mc_id=api_CatalogApi'}, {'title': 'Describe security and compliance concepts', 'url': 'https://learn.microsoft.com/en-us/training/modules/describe-security-concepts-methodologies/?WT.mc_id=api_CatalogApi'}, {'title': 'Create a voice calling web app with Azure Communication Services', 'url': 'https://learn.microsoft.com/en-us/training/modules/communication-services-voice-calling-web-app/?WT.mc_id=api_CatalogApi'}, {'title': 'The Principles of Sustainable Software Engineering', 'url': 'https://learn.microsoft.com/en-us/training/modules/sustainable-software-engineering-overview/?WT.mc_id=api_Catalog

In [44]:
print("Messages in next request:")
print(messages)
print()

second_response = client.chat.completions.create(
    messages=messages,
    model=deployment,
    tools=tools,
    tool_choice="auto",
    temperature=0,
)

print(second_response.choices[0].message)

Messages in next request:
[{'role': 'user', 'content': 'Find me a good course for a beginner student to learn Azure.'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Ml8ZFRZA8ZfFCUtAoIIGsfkF', function=Function(arguments='{"role":"student","product":"Azure","level":"beginner"}', name='search_courses'), type='function')], provider_specific_fields={'refusal': None}), {'role': 'tool', 'tool_call_id': 'call_Ml8ZFRZA8ZfFCUtAoIIGsfkF', 'content': "[{'title': 'Describe concepts of cryptography', 'url': 'https://learn.microsoft.com/en-us/training/modules/describe-concepts-of-cryptography/?WT.mc_id=api_CatalogApi'}, {'title': 'Host a web application with Azure App Service', 'url': 'https://learn.microsoft.com/en-us/training/modules/host-a-web-app-with-azure-app-service/?WT.mc_id=api_CatalogApi'}, {'title': 'Describe security and compliance concepts', 'url': 'https://l